# Open RAG From Scratch — Parts 1–4

This notebook builds the whole retrieval-augmented generation (RAG) chain, one cell at a time, using only open, self-hostable tools. There are no API keys — everything runs locally.

**How to read this notebook.** Each markdown cell explains one idea; the code cell right below it does exactly that and prints the result. Run a cell, look at what it produced, then read on. We build in four parts:

1. **Overview** — the whole chain end to end, in one go
2. **Indexing** — load → split → embed → store
3. **Retrieval** — find the chunks that matter for a question
4. **Generation** — turn those chunks into an answer

The ideas match the theory guide. The tools are all open:

| Stage | Open tool |
|---|---|
| Load a page | `requests` + `trafilatura` |
| Count tokens | HuggingFace `tokenizers` |
| Split text | `chonkie` |
| Embed text | `sentence-transformers` + Qwen3-Embedding-0.6B |
| Store vectors | `chromadb` |
| Generate | `ollama` |


## Setup

Install the open stack once. This uses your **global** environment — no virtualenv needed.

The one piece that is not pip-installed is **Ollama**, the local model runtime; install it separately and pull the model (`ollama pull gpt-oss:20b`).


In [1]:
# The dependencies live in requirements.txt and are already installed
# into this notebook's virtual environment (.venv), so there is nothing to
# install here. From a terminal, if you ever need to reinstall:
#     .venv\Scripts\python.exe -m pip install -r requirements.txt

### Imports and configuration

Everything tunable from the theory guide's *Parameters that matter* table lives in one `CONFIG` dict, so experiments never require editing the pipeline code.


In [5]:
import requests
import trafilatura
import chromadb
import numpy as np

from tokenizers import Tokenizer
from chonkie import RecursiveChunker
from sentence_transformers import SentenceTransformer
import ollama

print('imports ok')

imports ok


In [6]:
CONFIG = {
    'chunk_size_tokens': 300,
    'chunk_overlap_tokens': 50,
    'embed_model': 'Qwen/Qwen3-Embedding-0.6B',
    'store_path': './index/chroma',
    'collection': 'docs',
    'k': 5,
    'llm_model': 'gpt-oss:20b',
    'temperature': 0.0,
    'context_window': 32768,
    'max_answer_tokens': 1024,
}
CONFIG

{'chunk_size_tokens': 300,
 'chunk_overlap_tokens': 50,
 'embed_model': 'Qwen/Qwen3-Embedding-0.6B',
 'store_path': './index/chroma',
 'collection': 'docs',
 'k': 5,
 'llm_model': 'gpt-oss:20b',
 'temperature': 0.0,
 'context_window': 32768,
 'max_answer_tokens': 1024}

### Check the local model runtime

Generation needs a model server running on your machine. This cell confirms `ollama` is up and that the configured model has been pulled.


In [7]:
# Confirm the local runtime is reachable and the model is available.
try:
    listing = ollama.list()
    names = [m.get('name', m.get('model', '')) for m in listing.get('models', [])]
    print('Ollama is running. Available models:')
    for n in names:
        print(' -', n)
    present = any(CONFIG['llm_model'].split(':')[0] in n for n in names)
    print('Configured model present:', present)
    if not present:
        print('Pull it with:  ollama pull', CONFIG['llm_model'])
except Exception as e:
    print('Could not reach Ollama:', e)
    print('Start it with:  ollama serve   then   ollama pull', CONFIG['llm_model'])

Ollama is running. Available models:
 - gpt-oss:20b
Configured model present: True


# Part 1 — Overview

Before taking the chain apart, run the whole thing in a single cell so the shape of the system is clear. Read left to right: **load** a page, **split** it into chunks, **embed + store** them, **retrieve** the chunks relevant to a question, and **generate** an answer from them.

This is deliberately rough — later parts rebuild each step carefully with explanation.


In [8]:
# Load the embedding model once; it is reused for the rest of the notebook.
model = SentenceTransformer(CONFIG['embed_model'])

URL = 'https://lilianweng.github.io/posts/2023-06-23-agent/'
question = 'What is Task Decomposition?'

# ---- INDEX (offline) ----
text = trafilatura.extract(requests.get(URL).text)

chunker = RecursiveChunker(chunk_size=CONFIG['chunk_size_tokens'])
passages = [chunk.text for chunk in chunker.chunk(text)]

vectors = model.encode(passages, normalize_embeddings=True)

client = chromadb.PersistentClient(path=CONFIG['store_path'])
collection = client.get_or_create_collection(
    name=CONFIG['collection'], metadata={'hnsw:space': 'cosine'}
)
collection.upsert(
    ids=[f'overview:{i}' for i in range(len(passages))],
    documents=passages,
    embeddings=vectors.tolist(),
    metadatas=[{'chunk_index': i} for i in range(len(passages))],
)

# ---- RETRIEVE (online) ----
query_vector = model.encode([question], normalize_embeddings=True).tolist()
hits = collection.query(query_embeddings=query_vector, n_results=CONFIG['k'])
context = '\n\n'.join(hits['documents'][0])

# ---- GENERATE ----
prompt = f'''Answer the question based only on the following context:
{context}

Question: {question}
'''
reply = ollama.chat(
    model=CONFIG['llm_model'],
    options={'temperature': CONFIG['temperature']},
    messages=[{'role': 'user', 'content': prompt}],
)
print(reply['message']['content'])

{
    "thoughts": {
        "text": "Task decomposition is the process of breaking a complex task into smaller, manageable sub‑tasks or steps, enabling clearer planning, efficient execution, and easier problem solving.",
        "reasoning": "The context describes task decomposition as being performed by LLM prompting, task‑specific instructions, or human input, and emphasizes that a complicated task involves many steps that an agent must identify and plan ahead. It also notes that CoT transforms big tasks into multiple manageable tasks, highlighting the purpose of decomposition."
    }
}


That single cell is the whole of RAG. Notice the two halves: **indexing** happened once (load, split, embed, store), and only the last three steps — embed the question, retrieve, generate — run per question.

Now we take it apart, starting with indexing.

# Part 2 — Indexing

Indexing prepares your documents so they can be searched by meaning. It runs once, ahead of time, and has four steps: **load → split → embed → store**. The seven subsections below build them up one at a time.

We start small, with a single sentence, so the mechanics are easy to see.

## 2.1 Documents

A **document** is just the raw text you want the system to know about — a web page, a note, a file. At this stage it is text plus a little metadata (where it came from). We begin with the smallest possible example.


In [9]:
question = 'What kinds of pets do I like?'
document = 'My favorite pet is a cat.'

print('question:', question)
print('document:', document)

question: What kinds of pets do I like?
document: My favorite pet is a cat.


## 2.2 Token counting

Models do not read characters or whole words — they read **tokens** (word pieces). Counting tokens tells us how much text fits in the model's **context window**, and how big our chunks should be.

One rule matters: use the **model's own tokenizer**. A count from a different tokenizer will mis-size chunks. Here we load the tokenizer that belongs to our embedding model.


In [10]:
# Load the embedding model's own tokenizer and count with it.
tokenizer = Tokenizer.from_pretrained(CONFIG['embed_model'])

def count_tokens(text):
    return len(tokenizer.encode(text).ids)

print('tokens in document:', count_tokens(document))
print('tokens in question:', count_tokens(question))

tokens in document: 8
tokens in question: 9


## 2.3 Embeddings

An **embedding** turns text into a list of numbers — a vector — that captures its meaning. Texts with similar meaning end up as nearby vectors. This is what makes search semantic rather than a literal word match.

Each model always produces the same vector length; our model gives **1024 numbers**.


In [ ]:
# Embed both texts; each becomes a fixed-length vector.
document_vector = model.encode([document], normalize_embeddings=True)[0]
question_vector = model.encode([question], normalize_embeddings=True)[0]

print('vector length:', len(document_vector))
print('first 10 numbers:', np.round(document_vector[:10], 3))

vector length: 1024
first 8 numbers: [ 0.024  0.039  0.002 -0.079 -0.054  0.003  0.021  0.021  0.029  0.098]


## 2.4 Cosine similarity

To compare two embeddings we measure the **angle** between them — cosine similarity. A score near `1` means the texts are very similar in meaning; near `0` means unrelated. This is exactly how retrieval will pick the best chunks later.


In [17]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

score = cosine_similarity(question_vector, document_vector)
print('cosine similarity (question vs document):', round(score, 3))

cosine similarity (question vs document): 0.56


## 2.5 Document loaders

Real sources are messy — a web page is mostly menus, ads and footers wrapped around a small article. A **loader** fetches the page and keeps only the meaningful text, so that noise never reaches our embeddings.

`trafilatura` extracts the main article automatically. We also record a `source_id` so answers can cite where they came from.


In [13]:
source_id = 'lilianweng-agent'

raw_html = requests.get(URL).text
text = trafilatura.extract(raw_html)

print('characters extracted:', len(text))
print('preview:', text[:300].replace(chr(10), ' '), '...')

characters extracted: 39316
preview: Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it c ...


## 2.6 Splitting (chunking)

Long documents are cut into smaller **chunks**. A chunk is the unit that gets embedded, retrieved and shown to the model, so its size matters. Small chunks give precise matches; large chunks carry more context but dilute relevance. A small **overlap** keeps sentences from being sliced across a boundary.

Here we split by **tokens**, using the configured size of 300.


In [14]:
# Split the article into token-sized chunks.
chunker = RecursiveChunker(chunk_size=CONFIG['chunk_size_tokens'])
chunks = chunker.chunk(text)
passages = [chunk.text for chunk in chunks]

print('number of chunks:', len(passages))

number of chunks: 185


In [18]:
# Look at the first chunk and confirm its size.
first = passages[0]
print('tokens in first chunk:', count_tokens(first))
print('---')
print(first[:600])

tokens in first chunk: 49
---
Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. 


## 2.7 Vector store

A **vector store** saves each chunk together with its vector, forming a searchable library of our documents. We open a **persistent** Chroma collection (it lives on disk, so it survives restarts), then add the chunks.

The ids are stable (`<source_id>:<index>`), so re-running indexing updates entries instead of creating duplicates.


In [19]:
# Open (or create) the persistent collection.
client = chromadb.PersistentClient(path=CONFIG['store_path'])
collection = client.get_or_create_collection(
    name=CONFIG['collection'], metadata={'hnsw:space': 'cosine'}
)

print('items before adding:', collection.count())

items before adding: 185


In [20]:
# Embed every chunk and store it with its metadata.
vectors = model.encode(passages, normalize_embeddings=True)

collection.upsert(
    ids=[f'{source_id}:{i}' for i in range(len(passages))],
    documents=passages,
    embeddings=vectors.tolist(),
    metadatas=[{'source_id': source_id, 'chunk_index': i} for i in range(len(passages))],
)

print('items after adding:', collection.count())

items after adding: 370


# Part 3 — Retrieval

Now the online half begins. To answer a question we embed it with the **same** embedding model, then ask the store for the chunks whose vectors are nearest — the "top-k".

`k` is a trade-off: too few and we may miss the answer; too many and we flood the model with loosely related text.


In [21]:
query = 'What is Task Decomposition?'
query_vector = model.encode([query], normalize_embeddings=True).tolist()

hits = collection.query(query_embeddings=query_vector, n_results=CONFIG['k'])

for i, (cid, dist) in enumerate(zip(hits['ids'][0], hits['distances'][0]), start=1):
    print(f'[{i}] id={cid}  score={1 - dist:.3f}')

[1] id=overview:12  score=0.778
[2] id=lilianweng-agent:12  score=0.778
[3] id=overview:9  score=0.721
[4] id=lilianweng-agent:9  score=0.721
[5] id=lilianweng-agent:7  score=0.697


In [22]:
# Show the actual passages that were retrieved.
for i, passage in enumerate(hits['documents'][0], start=1):
    print(f'--- chunk [{i}] ---')
    print(passage[:500])
    print()

--- chunk [1] ---
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.


--- chunk [2] ---
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.


--- chunk [3] ---
CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.


--- chunk [4] ---
CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.


--- chunk [5] ---
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition




Those are the chunks the model will be allowed to use. If the answer is not in them, no model can invent it — which is why **retrieval quality** matters more than model size.

# Part 4 — Generation

The last stage. The retrieved chunks go into a **prompt** next to the question, and the language model writes an answer **from that context** rather than from memory. This grounding is the "G" in RAG.

The prompt asks for three things: answer only from the context, cite sources, and admit when the answer is not present.


In [23]:
# Build the prompt: retrieved context + the question.
context = '\n\n'.join(hits['documents'][0])

prompt = f'''Answer the question based only on the following context:
{context}

Question: {query}
'''
print(prompt[:1200])

Answer the question based only on the following context:
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.


Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.


CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.


CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.


A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition


Question: What is Task Decomposition?



In [24]:
# Generate the answer with the local model.
reply = ollama.chat(
    model=CONFIG['llm_model'],
    options={'temperature': CONFIG['temperature']},
    messages=[{'role': 'user', 'content': prompt}],
)
answer = reply['message']['content']
print(answer)

**Task Decomposition** is the process of breaking a large, complex task into a set of smaller, manageable sub‑tasks or steps.  
- It can be performed by a language model (LLM) using simple prompts such as “Steps for XYZ.\n1.” or “What are the subgoals for achieving XYZ?”  
- It can also be guided by task‑specific instructions (e.g., “Write a story outline.” for a novel).  
- Human input can be used to define or refine the decomposition.

By decomposing a task, an agent gains a clear list of actions it must take and can plan ahead. This approach is closely related to Chain‑of‑Thought (CoT) reasoning, which also transforms big tasks into multiple manageable steps and illuminates the model’s internal reasoning process.


In [25]:
# Map the citations back to their sources.
for i, (cid, meta, dist) in enumerate(
    zip(hits['ids'][0], hits['metadatas'][0], hits['distances'][0]), start=1
):
    sid = meta.get('source_id')
    print(f'[{i}] score={1 - dist:.3f}  source={sid}  id={cid}')

[1] score=0.778  source=None  id=overview:12
[2] score=0.778  source=lilianweng-agent  id=lilianweng-agent:12
[3] score=0.721  source=None  id=overview:9
[4] score=0.721  source=lilianweng-agent  id=lilianweng-agent:9
[5] score=0.697  source=lilianweng-agent  id=lilianweng-agent:7


# Part 5 — Putting it together

Everything above collapses into one plain function, exactly as the theory guide promised: `rag(question) -> answer`. Indexing stays as it is; the function does the serving half — embed, retrieve, prompt, generate.


In [26]:
def rag(question: str, k: int = CONFIG['k']) -> str:
    query_vector = model.encode([question], normalize_embeddings=True).tolist()
    hits = collection.query(query_embeddings=query_vector, n_results=k)
    context = '\n\n'.join(hits['documents'][0])
    prompt = f'''Answer the question based only on the following context:
{context}

Question: {question}
'''
    reply = ollama.chat(
        model=CONFIG['llm_model'],
        options={'temperature': CONFIG['temperature']},
        messages=[{'role': 'user', 'content': prompt}],
    )
    return reply['message']['content']

In [28]:
for q in [
    'What is Task Decomposition?',
    'How does reflection help an agent?',
    'What is a plan-and-execute agent?',
    'Who is Msawenkosi Magwaza?'
]:
    print('Q:', q)
    print(rag(q))
    print('-' * 80)

Q: What is Task Decomposition?
**Task Decomposition** is the process of breaking a complex or large task into a series of smaller, manageable sub‑tasks or steps. By doing so, an agent (or a language model) can:

1. **Identify the individual actions** needed to reach the overall goal.  
2. **Plan ahead** by ordering those actions logically.  
3. **Execute** each step more reliably, often with clearer guidance or instructions.

Task decomposition can be achieved in several ways:

- **LLM prompting**: Simple prompts such as “Steps for XYZ.\n1.” or “What are the subgoals for achieving XYZ?” let the model generate a list of steps.  
- **Task‑specific instructions**: Providing a clear directive like “Write a story outline.” prompts the model to produce a structured outline.  
- **Human inputs**: A human can supply or refine the sub‑tasks, ensuring they align with real‑world constraints or preferences.

In short, task decomposition turns a big, often ambiguous goal into a sequence of concrete

That is the complete chain, built one cell at a time. From here, the natural improvements are better **retrieval** (reranking, hybrid search), better **chunking**, or a larger model — in that order of impact.